# FHRS Rating Banding Modelling

This notebook details regression model builds and testing for FSA Hygiene Rating bandings. Initial decisions / approach are below:

#### Rating Bandings

As decided in `./01_data_exploration.ipynb`, I will initially target a three-tier banding (`critical`/`fail`/`pass`), with a binary fallback (`<=2*`, or `<5*`) if the first approach doesn't hold up.  

#### Features

As decided in `./04_imd_domain_join.ipynb`, I will use two LSOA-linked (location) deprivation factors: `income_decile` and `crime_decile`, from the domain-level IMD 2025 dataset. Overall `imd_decile` will be used as a fallback if needed.   

From the raw dataset, I have chosen to include:
- `BusinessType` - a clear candidate and a natural feature, with obvious impacts on an establishment's operation and therefore risk profile.  
- `LocalAuthorityName` - while this will already be somewhat represented through the deprivation deciles, local factors within a borough can vary wildly. For example, Tower Hamlets contains extremes in Canary Wharf (affluent, highly developed) and Whitechapel (among the most deprived neighbourhoods in London). Different borough Local Authorities may have different Food Safety initiatives or support, which again could impact likely ratings.    

#### Class-weight balancing

The requirement for class-weight balancing during modelling was identified during data cleaning, where the critical tier was found to represent only ~5.6% of gradable establishments London-wide.

#### Regression Model Type

I will use a **standard multinomial logistic regression** as my initial model: this is a relatively simple, well-proven and well-documented model, with built-in class balancing support (`sklearn`'s `LogisticRegression` has `class_weight="balanced"`). Notably, it treats the three rating tiers as unordered categories. The need for class weight balancing was identified in the initial data exploration notebook.   

An alternative (possibly secondary) approach is to use an **ordinal logistic regression**: this treats the three tiers as having an order (`critical < fail < pass`), in line with the real-world application of FHRS rating bands. While this is perhaps the more 'statistically correct' fit, the model assumes that each feature's effect is the same size at every threshold, e.g. the Income decile affects the critical -> fail boundary by the same magnitude as the fail -> pass boundary; this is not guaranteed to be true. In addition, `statsmodels`' `OrderedModel` may not handle class weighting as cleanly, which may in turn require manual workarounds.  

On the basis that a standard multinomial is 'safer' from a class-weight balancing aspect, I will use this approach first to get a working baseline, and will then consider trying (and comparing) an ordinal regression afterwards.  

### Data loading & modelling population build

First I will load and check the latest dataset. 

I will then extract the modelling population (establishments with a gradable rating and matching IMD-domain decile data) and build the required additional rating / rating tier columns:

In [1]:
# load data, build modelling DF
import pandas as pd

df = pd.read_csv(
    "../data/processed/fsa_london_establishments_with_domains.csv",
    dtype={"PostCode": str, "postcode_tier": str, "lsoa21cd": str, "RightToReply": str}
)

print(df.shape) # should be (81217, 40)

# same population as the domains notebook: gradable rating + matched deprivation data
model_df = df[
    df["RatingValue"].isin(["0","1","2","3","4","5"]) &
    df["income_decile"].notna() &
    df["crime_decile"].notna()
].copy()

# add required columns
model_df["rating_int"] = model_df["RatingValue"].astype(int)
model_df["tier"] = pd.cut(
    model_df["rating_int"],
    bins=[-1, 2, 4, 5],
    labels=["critical", "fail", "pass"],
)

print(len(model_df)) # should be 63,409
print(model_df["tier"].value_counts())

(81217, 40)
63409
tier
pass        42331
fail        17215
critical     3863
Name: count, dtype: int64


-> expected values confirming there has been no drift.  

I'll check category sizes again for `BusinessType` and `LocalAuthorityName` before turning them into dummy columns, in order to see how thin the rarest categories are within the _modelling-specific_ population:  

In [2]:
# check category sizes for one-hot encoding dummy cols
print(model_df["BusinessType"].value_counts())
print()
print(model_df["LocalAuthorityName"].value_counts().tail(10))

BusinessType
Restaurant/Cafe/Canteen                  23998
Retailers - other                        12650
Takeaway/sandwich shop                    8240
Caring Premises                           4057
Pub/bar/nightclub                         3663
School/college/university                 3165
Other catering premises                   2664
Retailers - supermarkets/hypermarkets     2219
Hotel/bed & breakfast/guest house          926
Mobile caterer                             737
Manufacturers/packers                      640
Distributors/Transporters                  355
Importers/Exporters                         84
Farmers/growers                             11
Name: count, dtype: int64

LocalAuthorityName
Haringey                1614
Redbridge               1505
Havering                1360
Harrow                  1330
Bexley                  1307
Richmond-Upon-Thames    1233
Kingston-Upon-Thames    1139
Sutton                  1059
Merton                  1041
Barking and Dagenham  

-> Local Authority shows no issues, however when looking at Business Type:

The tail of this category is a concern, with the low counts for Farmers/growers (11 rows), Importers/Exporters (84 rows) and Distributors/Transporters (355 rows) likely to yield zero or near-zero examples of establishments in the 'critical' tier.  

What this potentially means is that the model can't estimate a stable coefficient from a dummy column, resulting in unstable or exaggerated results.  

In order to define a cutoff:

In [3]:
# check Business Type tier representation:
tier_by_type = pd.crosstab(model_df["BusinessType"], model_df["tier"])
print(tier_by_type)

tier                                   critical  fail   pass
BusinessType                                                
Caring Premises                              79   580   3398
Distributors/Transporters                    11    63    281
Farmers/growers                               0     4      7
Hotel/bed & breakfast/guest house            26   150    750
Importers/Exporters                           4    16     64
Manufacturers/packers                        39   138    463
Mobile caterer                               38   201    498
Other catering premises                      93   575   1996
Pub/bar/nightclub                           138   787   2738
Restaurant/Cafe/Canteen                    1490  6624  15884
Retailers - other                          1203  4487   6960
Retailers - supermarkets/hypermarkets        57   313   1849
School/college/university                    33   308   2824
Takeaway/sandwich shop                      652  2969   4619


-> tier counts by Business Type confirms this risk

In order to defend agaisnst this risk, I will look to collapse rare categories into an 'Other' banding where a minimum critical rating (i.e. the rarest tier) count is not met:

In [4]:
MIN_CRITICAL_COUNT = 10 # in line with loose 'events per variable' rule

critical_counts = tier_by_type["critical"] # count critical ratings, not total rows
rare_types = critical_counts[critical_counts < MIN_CRITICAL_COUNT].index.tolist() 
    # those that do not meet the threshold set above

print(f"Business types to collapse into 'Other': {rare_types}")

# new DF from original, but swaps BusinessType value with "Other" when in rare_types
model_df["BusinessTypeGrouped"] = model_df["BusinessType"].replace(
    dict.fromkeys(rare_types, "Other")
        # assemble dict: {"Farmers/growers": "Other", "Importers/Exporters": "Other", ...}
    # .replace() then takes assembled dict and any keys found in BusinessType are replaced
)

print(model_df["BusinessTypeGrouped"].value_counts())

Business types to collapse into 'Other': ['Farmers/growers', 'Importers/Exporters']
BusinessTypeGrouped
Restaurant/Cafe/Canteen                  23998
Retailers - other                        12650
Takeaway/sandwich shop                    8240
Caring Premises                           4057
Pub/bar/nightclub                         3663
School/college/university                 3165
Other catering premises                   2664
Retailers - supermarkets/hypermarkets     2219
Hotel/bed & breakfast/guest house          926
Mobile caterer                             737
Manufacturers/packers                      640
Distributors/Transporters                  355
Other                                       95
Name: count, dtype: int64
